In [26]:
import psycopg2
import pandas as pd
# pip install psycopg2-binary
#https://adsai.buas.nl/Year1/BlockD/assets/code_book.html

db_params = {
    'host': '194.171.191.226',
    'port': '6379',
    'database': 'postgres',
    'user': 'group15',
    'password': 'blockd_2024group15_73'
}

con = psycopg2.connect(**db_params)

In [2]:
sql = '''
SELECT *
FROM data_lake.accident_data_17_23
'''
cur = con.cursor()
cur.execute(sql)

# group15_warehouse.name = DataFrame.to_sql(name_of_df)
# SELECT *
# FROM grup15_warehouse.name

data = cur.fetchall()

columns_names = [col.name for col in cur.description]

cur.close()

pd.set_option('display.max_columns', None)
df = pd.DataFrame(data, columns = columns_names)
display(df.head(10))

,Year,Accident severity,municipality,town,First Mode of Transport,Second mode of Transport,Area Type,Light condition,Road Location,Road condition,Road surface,Road situation,Speed limit,street,weather,accidents
0,2017,Fatal,Breda,BREDA,Car,Pedestrian,Urban area,Darkness,Intersection,Wet/damp,Brick,Bend,30 km/h,Valkeniersplein,Rain,1
1,2017,Fatal,Breda,BREDA,Lorry,Other,Urban area,Daylight,Intersection,Wet/damp,Brick,Intersection - 4 arms,50 km/h,Markendaalseweg,Dry,1
2,2017,Fatal,Breda,BREDA,Lorry,Other,Urban area,Daylight,Road section,Dry,Asphalt (other),Straight road,50 km/h,Academiesingel,Dry,1
3,2017,Injured,Breda,BAVEL,Car,Lorry,Rural area,Darkness,Road section,Wet/damp,Asphalt (other),Bend,120 km/h,KP ST.ANNABOSCH,Dry,1
4,2017,Injured,Breda,BAVEL,Car,Other,Rural area,Darkness,Road section,Wet/damp,Porous asphalt,Straight road,130 km/h,RYKSWG,Rain,1
5,2017,Injured,Breda,BAVEL,Delivery van,Car,Rural area,Darkness,Road section,Wet/damp,Asphalt (other),Straight road,80 km/h,Gilzeweg,Rain,1
6,2017,Injured,Breda,BREDA,Bicycle,Motorcycle,Urban area,Daylight,Intersection,Dry,Asphalt (other),Intersection - 4 arms,50 km/h,Nieuwe Kadijk,Dry,1
7,2017,Injured,Breda,BREDA,Bicycle,Other,Urban area,Daylight,Intersection,Dry,Asphalt (other),Intersection - 4 arms,30 km/h,Rithsestraat,Dry,1
8,2017,Injured,Breda,BREDA,Bicycle,Other,Urban area,Daylight,Intersection,Dry,Brick,Straight road,Unknown,Steenbroek,Dry,1
9,2017,Injured,Breda,BREDA,Bicycle,Other,Unknown,Darkness,Road section,Unknown,Unknown,Unknown,50 km/h,Achter Emer,Unknown,1


In [3]:
df = df[df['town'] == 'BREDA']

df = df.groupby('street').size().reset_index(name='no_of_accidents')

display(df.head(10))

,street,no_of_accidents
0,'t Zand,1
1,Aardenhoek,3
2,Abeelstraat,4
3,Acaciastraat,3
4,Academiesingel,37
5,Achter Emer,8
6,Achter de Lange Stallen,2
7,Achtererf,5
8,Adriaan Klaassenstraat,2
9,Adriaan van Bergenstraat,9


Creating SQL query and uploading it to the server

In [4]:
sql = '''
CREATE TABLE IF NOT EXISTS group15_warehouse.accidents_on_each_streetAS
SELECT 
    street, 
    COUNT(*) AS no_of_accidents
FROM 
    data_lake.accident_data_17_23
WHERE 
    town = 'BREDA'
GROUP BY 
    street
'''

cur = con.cursor()
cur.execute(sql)

con.commit()

cur.close()

Downloading own table from the server

In [5]:
sql = '''
SELECT *
FROM group15_warehouse.accidents_on_each_street;
'''

cur = con.cursor()
cur.execute(sql)

data = cur.fetchall()
cur.close()

columns_names = [col.name for col in cur.description]
df = pd.DataFrame(data, columns = columns_names)

display(df.head(5))

,street,no_of_accidents
0,Emerparklaan,79
1,Thorbeckeplein,2
2,Pietersberg,1
3,Hamdijk,12
4,Oude Vest,22


Importing Keijs dataset from GitHub

In [6]:
import pandas as pd 

import folium
 
url = "https://raw.githubusercontent.com/KeesKlijs/dataset/main/accidents_data_breda_2013-2022.csv"
 
df = pd.read_csv(url)
 
df.head()

,VKL_NUMMER,Year,Accident_Severity,Parties_Involved,Area_Code,Latitude,Longitude
0,20140141297,2014,Vehicle Damage,1,JTE0223195056,51.568960,4.764808
1,20140141298,2014,Vehicle Damage,2,HTT02142110100559,51.627604,4.704797
2,20140141302,2014,Vehicle Damage,2,WVK0221202005,51.600120,4.748510
3,20140141304,2014,Vehicle Damage,2,WVK0225198149,51.582111,4.777310
4,20140141305,2014,Wounded,3,JTE0228205006,51.613091,4.799947


Coordinates normalization.

In [7]:
decimal_precision = 3 # about 111 meters

df['latitude'] = df['Latitude'].round(decimal_precision)
df['longitude'] = df['Longitude'].round(decimal_precision)

df = df.groupby(['latitude', 'longitude']).size().reset_index(name='no_of_accidents')

display(df.head(10))

,latitude,longitude,no_of_accidents
0,51.486,4.734,2
1,51.486,4.736,2
2,51.487,4.736,4
3,51.488,4.736,4
4,51.488,4.737,5
5,51.489,4.737,4
6,51.489,4.738,1
7,51.490,4.738,2
8,51.490,4.739,15
9,51.490,4.740,1


In [8]:
df1 = df

Adding column with streets names and leaving only coordinates with the most of accidents

In [9]:
import geocoder

def get_street_name(row):
    location = geocoder.osm([row['latitude'], row['longitude']], method='reverse')
    if location.ok:
        return location.street
    else:
        return None

df = df1.sort_values(by='no_of_accidents', ascending=False)
df = df.head(10)

a = []

for k in range(10):
    a.append(get_street_name(df.iloc[k]))

df['street'] = a

display(df)

INFO: Requested https://nominatim.openstreetmap.org/search?q=51.569%2C+4.765&format=jsonv2&addressdetails=1&limit=1
INFO: Requested https://nominatim.openstreetmap.org/search?q=51.6%2C+4.74&format=jsonv2&addressdetails=1&limit=1
INFO: Requested https://nominatim.openstreetmap.org/search?q=51.564%2C+4.766&format=jsonv2&addressdetails=1&limit=1
INFO: Requested https://nominatim.openstreetmap.org/search?q=51.6%2C+4.749&format=jsonv2&addressdetails=1&limit=1
INFO: Requested https://nominatim.openstreetmap.org/search?q=51.582%2C+4.743&format=jsonv2&addressdetails=1&limit=1
INFO: Requested https://nominatim.openstreetmap.org/search?q=51.602%2C+4.722&format=jsonv2&addressdetails=1&limit=1
INFO: Requested https://nominatim.openstreetmap.org/search?q=51.605%2C+4.725&format=jsonv2&addressdetails=1&limit=1
INFO: Requested https://nominatim.openstreetmap.org/search?q=51.601%2C+4.782&format=jsonv2&addressdetails=1&limit=1
INFO: Requested https://nominatim.openstreetmap.org/search?q=51.59%2C+4.759&f

,latitude,longitude,no_of_accidents,street
568,51.569,4.765,290,Schubertlaan
1856,51.600,4.740,162,Veldsteen
447,51.564,4.766,76,Burgemeester de Manlaan
1860,51.600,4.749,71,Emerparklaan
1058,51.582,4.743,64,Ettensebaan
1939,51.602,4.722,59,A16
2055,51.605,4.725,50,Walpad
1915,51.601,4.782,46,Nieuwe Kadijk
1437,51.590,4.759,42,Lunetstraat
1951,51.602,4.770,42,Backer en Ruebweg


Uploading df to the server

In [27]:
import psycopg2

cur = con.cursor()

try:
    create_table_query = """
    CREATE TABLE IF NOT EXISTS group15_warehouse.dangerous_coordinates (
        latitude FLOAT,
        longitude FLOAT,
        no_of_accidents INT,
        street TEXT
    );
    """
    cur.execute(create_table_query)

    con.commit()

    for index, row in df.iterrows():
        query = """
        INSERT INTO group15_warehouse.dangerous_coordinates(latitude, longitude, no_of_accidents, street)
        VALUES (%s, %s, %s, %s);
        """
        values = (row['latitude'], row['longitude'], row['no_of_accidents'], row['street'])
        cur.execute(query, values)

    con.commit()

except Exception as e:
    print("Error:", e)
    con.rollback()

finally:
    print('data uploaded')
    cur.close()

data uploaded


In [28]:
sql = '''
SELECT *
FROM group15_warehouse.dangerous_coordinates;
'''

cur = con.cursor()
cur.execute(sql)

data = cur.fetchall()
cur.close()

columns_names = [col.name for col in cur.description]
df = pd.DataFrame(data, columns = columns_names)

display(df.head(5))

,street,no_of_accidents
0,Emerparklaan,79
1,Thorbeckeplein,2
2,Pietersberg,1
3,Hamdijk,12
4,Oude Vest,22
